In [2175]:
import re
import json
from bs4 import BeautifulSoup

## Open the json and fill a dictionary

In [2176]:
def open_monsters():
    with open('./data/monsters_pretty.json', 'r', encoding="utf-8") as f_:
        monsters_dict = json.load(f_)
    return monsters_dict

monsters_dict = open_monsters()

In [2177]:

monster_text = monsters_dict["outsiders_demon_balor"]
print(monster_text)


<html>
 <head>
 </head>
 <body>
  <div class="article-content" id="article-content">
   <div class="breadcrumbs">
    <a class="bread-parent" href="https://www.d20pfsrd.com/" parentid="52278" title="Home">
     Home
    </a>
    &gt;
    <a class="bread-parent" href="https://www.d20pfsrd.com/bestiary/" parentid="57186" title="Bestiary">
     Bestiary
    </a>
    &gt;
    <a class="bread-parent" href="https://www.d20pfsrd.com/bestiary/monster-listings/" parentid="57428" title="(Bestiary) By Type">
     (Bestiary) By Type
    </a>
    &gt;
    <a class="bread-parent" href="https://www.d20pfsrd.com/bestiary/monster-listings/outsiders/" parentid="62127" title="Outsiders">
     Outsiders
    </a>
    &gt;
    <a class="bread-parent" href="https://www.d20pfsrd.com/bestiary/monster-listings/outsiders/demon/" parentid="62468" title="Demons">
     Demons
    </a>
    &gt;
   </div>
   <h1>
    Demon, Balor
   </h1>
   <script type="text/javascript">
    ognCreateVideoAdSpotOutstream("nitropay-

## remove top lines up to <h1>

In [2178]:
top_patterns = [r'^.*?(?=<h1)',
                r'<script[^>]*>.*?</script>',
                r'<div class="toc_light_blue[^>]*>.*?</div>'
                ]

def scrub(monsters_dict, scrub_patterns):
    for monster in monsters_dict:
        for pattern in scrub_patterns:
            monsters_dict[monster] = re.sub(pattern, '', monsters_dict[monster], flags=re.DOTALL)

scrub(monsters_dict, top_patterns)



print(monsters_dict["outsiders_demon_balor"])

<h1>
    Demon, Balor
   </h1>
   
   
   <div class="ogn-childpages" style="float:right">
    <p>
     <b>
      Subpages
     </b>
    </p>
    <ul class="ogn-childpages">
     <li class="page new parent" post_id="62472">
      <a href="https://www.d20pfsrd.com/bestiary/monster-listings/outsiders/demon/balor/balor-lord/">
       Demon, Balor Lord
      </a>
     </li>
    </ul>
   </div>
   <div class="statblock">
    <p class="description">
     This winged fiend’s horned head and fanged visage present the perfection of the demonic form, fire spurting from its flesh.
    </p>
    <p class="title">
     Balor
     <span class="level">
      CR 20
     </span>
    </p>
    <p>
     <b>
      XP 307,200
     </b>
     <br/>
     CE Large
     <a href="https://www.d20pfsrd.com/bestiary/rules-for-monsters/creature-types#Outsider" rel="nofollow">
      outsider
     </a>
     (
     <a href="https://www.d20pfsrd.com/bestiary/rules-for-monsters/creature-types#TOC-Chaotic" rel="nofollow">
 

## Check if it is a category or monster

In [2179]:
def check_has_child(monsters_dict):
    has_child = []
    no_child = []
    for entry in monsters_dict:
        match = re.search(r'<div class="ogn-childpages"', monsters_dict[entry])
        if match:
            has_child.append(entry)
        else:
            no_child.append(entry)
    return has_child, no_child

has_child, no_child = check_has_child(monsters_dict)

print(f"{len(has_child)} {len(no_child)}")

436 5641


In [2180]:
def check_if_monster(has_child):
    has_child_has_stats = []
    has_child_no_stats = []
    for entry in has_child:
        match = re.search(r'<p class="divider">\s*STATISTICS\s*</p>', monsters_dict[entry])
        # match = re.search(r'<div class="statblock">', monsters_dict[entry])
        if match:
            has_child_has_stats.append(entry)
        else:
            has_child_no_stats.append(entry)
    return has_child_has_stats, has_child_no_stats

has_child_has_stats, has_child_no_stats = check_if_monster(has_child)

# print(monsters_dict[has_child_no_stats[1]])
# print(f"{len(has_child_has_stats)} {len(has_child_no_stats)}")
print(f" no child {len(no_child)} \n has child, no stats {len(has_child_has_stats)} \n has child has stats {len(has_child_no_stats)}")


 no child 5641 
 has child, no stats 219 
 has child has stats 217


In [2181]:

# categories_dict = {}
# for entry in has_child_no_stats:
#     category = monsters_dict.pop(entry)
#     categories_dict.update({entry: category})

# print(len(categories_dict))
# print(len(monsters_dict))

def split_dict(original_dict, split_list):
    split_dict = {}
    for entry in split_list:
        split_entry = original_dict.pop(entry)
        split_dict.update({entry: split_entry})
    print(f'original: {len(original_dict)}')
    print(f'split: {len(split_dict)}')
    return original_dict, split_dict

monsters_dict, categories_dict = split_dict(monsters_dict, has_child_no_stats)


original: 5860
split: 217


## Remove Childpages Div

In [2182]:
scrub_patterns = [r'<div class="ogn-childpages" style="float:right">.*?</div>', r'<div class="product-right">.*?</div>']

scrub(monsters_dict, scrub_patterns)
print(len(monsters_dict))

5860


## check if statblock

In [2183]:
statblock_pattern = r'<div class="statblock">(.*?)</div>'
thirdpp_pattern = r'3pp'
description_pattern = r'<p class="description">.*?<p>'

def check_content(monsters_dict, pattern):
    yes = []
    no = []
    for monster in monsters_dict:
        monster_text = monsters_dict[monster]
        match = re.search(pattern, monster_text, flags=re.DOTALL | re.IGNORECASE)
        if match:
            yes.append(monster)
        else:
            no.append(monster)
    return yes, no


third_party, vanilla = check_content(monsters_dict, thirdpp_pattern)
print("3pp y/n:", len(third_party), len(vanilla))

yes_statblock, no_statblock = check_content(monsters_dict, statblock_pattern)
print("statblock y/n:", len(yes_statblock), len(no_statblock))

yes_description, no_description = check_content(monsters_dict, description_pattern)
print("description y/n:", len(yes_description), len(no_description))

3pp y/n: 494 5366
statblock y/n: 3215 2645
description y/n: 4006 1854


## try to Identify the Title/Header/Creature block

In [2184]:
title_header_pattern = r'<p class="(?:title|header|creature)".*?(<span(?: class="(?:level|creature-level)")?)?.*?</p>'
h1_pattern = r'<h1>.*?</h1>'

yes, no = check_content(monsters_dict, h1_pattern)
print("h1 y/n:", len(yes), len(no))
yes_head, no_head = check_content(monsters_dict, title_header_pattern)
print("title/header y/n:", len(yes_head), len(no_head))
monsters_dict, monsters_table_dict = split_dict(monsters_dict, no_head)








h1 y/n: 5860 0
title/header y/n: 5747 113
original: 5747
split: 113


In [2185]:
for monster in monsters_dict:
    match = re.search(r'<p class="header">\s*APPEARANCE\s*</p>', monsters_dict[monster], re.DOTALL)
    if match:
        print(monster)
        re.sub(r'<p class="header">\s*APPEARANCE\s*</p>', '', monsters_dict[monster], re.DOTALL)

monstrous-humanoids_hag-mute
plants_cauldron-bloom-2
fey_duende
fey_bulabar-2
outsiders_devil_devil-suffragan
monstrous-humanoids_hag-ash
fey_mythic-dryad
fey_elananx
fey_brownie-2
outsiders_dream-spectre-2
fey_fey-wolf
fey_cold-rider-2
fey_fear-eater-2
fey_blodeuwedd-2
outsiders_hag-dreamthief
fey_summer-hora-queen
fey_gremlin-erinat
fey_choxani-2
fey_mythic-fossegrim
undead_cyhyraeth
fey_mythic-gremlin-jinkin
fey_gremlin-hobkins
humanoids_angurboda
fey_hora-summer
fey_mythic-forlarren
fey_horzitoth
outsiders_daemons_daemon-venedaemon-2
fey_fey-courtier
fey_escorite-2
constructs_guardian-doll-2
fey_mythic-fastachee
fey_fey-grizzly-bear
aberrations_mythic-fachen
fey_draxie
fey_bogeyman-2
undead_danse-macabre-2
outsiders_azata-yamah
undead_death-coach-2
fey_buckawn-2
fey_mythic-gremlin-fuath
outsiders_daemons_daemon-harbinger-of-broken-deals-fine-print-and-unfair-bargains
fey_mythic-gremlin-nuglub
outsiders_devil_devil-joyful-thing
fey_crossroads-guardian
fey_buckawn-gang-leader
fey_gum

## split off Mythic creatures

In [2186]:
mythic_list = []

for monster in monsters_dict:
    head = re.search(r'<p class="(?:title|header|creature)".*?(<span(?: class="(?:level|creature-level)")?)?.*?</p>', monsters_dict[monster], re.DOTALL).group(0)
    if re.search(r'Mythic', head) and re.search(r'MR', head):
        mythic_list.append(monster)

monsters_dict, monsters_mythic_dict = split_dict(monsters_dict, mythic_list)

original: 5586
split: 161


## Summary

In [2187]:
print(len(monsters_dict),len(monsters_mythic_dict),len(monsters_table_dict),len(categories_dict) )

5586 161 113 217


# Parsing

In [2188]:
print(next(iter(monsters_dict.values())))

<h1>
    Gargoyle, Green Guardian (3pp)
   </h1>
   
   
   </div>
   <p class="description">
    This winged humanoid creature is carved of a strange green stone with eyes rich black in color.
   </p>
   <p class="title">
    Green Guardian
    <br/>
    <span>
     CR 4
    </span>
   </p>
   <p>
    <b>
     XP 1,200
    </b>
    <br/>
    CE Medium
    <a href="https://www.d20pfsrd.com/bestiary/rules-for-monsters/creature-types#TOC-Monstrous-Humanoid">
     monstrous humanoid
    </a>
    (
    <a href="https://www.d20pfsrd.com/bestiary/rules-for-monsters/creature-types#TOC-Earth">
     earth
    </a>
    )
    <br/>
    <b>
     Init
    </b>
    +2;
    <b>
     Senses
    </b>
    <a href="https://www.d20pfsrd.com/gamemastering/special-abilities#TOC-Darkvision">
     darkvision
    </a>
    60 ft.,
    <a href="https://www.d20pfsrd.com/gamemastering/special-abilities#TOC-Low-Light-Vision">
     low-light vision
    </a>
    ;
    <a href="https://www.d20pfsrd.com/skills/percepti

#### save h1 and description

In [2189]:
h1_pattern = r'<h1>(.*?)</h1>'

monsters_data = {}

for monster in monsters_dict:
    match = re.search(h1_pattern, monsters_dict[monster], re.DOTALL)
    monsters_data.update({monster: {"title1": match.group(1).strip(),
                                    "title2": monster.split("_")[-1]}})
    monsters_dict[monster] = re.sub(r'<h1>.*?</h1>', '', monsters_dict[monster], flags=re.DOTALL).strip()

print(monsters_data["constructs_golem_golem-wax"])

{'title1': 'Golem, Wax', 'title2': 'golem-wax'}


In [2190]:
for monster in monsters_dict:
    match = re.search(r'<p class="description">(.*?)</p>', monsters_dict[monster], flags=re.DOTALL)
    if match:
        desc = match.group(1).strip()
        desc = re.sub(r'<[^>]+>', '', desc).strip()
        desc = re.sub(r'\s+', ' ', desc).strip()
        monsters_data[monster]["desc_short"] = desc
        monsters_dict[monster] = re.sub(r'<p class="description">.*?</p>', '', monsters_dict[monster], flags=re.DOTALL).strip()
    else:
        monsters_data.update({monster: {"desc_short": ""}})
    monsters_dict[monster] = re.sub(
        r'^.*?(?=<p class="(?:title|header|creature)")',
        '', 
        monsters_dict[monster], 
        flags=re.DOTALL
    )

#### save source and remove trail

In [2191]:
for monster in monsters_dict:
    match = re.search(r'Copyright.*$', monsters_dict[monster], flags=re.DOTALL | re.IGNORECASE)
    if match:
        # print(match.group(0))
        match_href = re.search(r'<a(.*?)href=.*?>(.*?)</a>', match.group(0), flags=re.DOTALL)
        match_p = re.search(r'<p.*?>(.*?)</p>', match.group(0), flags=re.DOTALL)
        match_i = re.search(r'<i.*?>(.*?)</i>', match.group(0), flags=re.DOTALL)
        match_div = re.search(r'<div.*?>(.*?)</div>', match.group(0), flags=re.DOTALL)
        if match_href:
            monsters_data[monster]['source'] = match_href.group(1).strip()
        elif match_p:
            source = re.sub(r'<[^>]+>', '', match_p.group(1)).strip()
            source = re.sub(r'\s+', ' ', source).strip()
            # print(source)
        elif match_i:
            source = re.sub(r'<[^>]+>', '', match_i.group(1)).strip()
            source = re.sub(r'\s+', ' ', source).strip()
        elif match_div:
            source = re.sub(r'<[^>]+>', '', match_div.group(1)).strip()
            source = re.sub(r'\s+', ' ', source).strip()
            monsters_data[monster]['source'] = source
        else:
            monsters_data[monster]['source'] = ""   
    else:
        monsters_data[monster]['source'] = ""


        

In [2192]:
for monster in monsters_dict:
    monsters_dict[monster] = re.sub(
        r'<!--div style="clear:both">.*$',
        '', 
        monsters_dict[monster], 
        flags=re.DOTALL
    )
    # print(repr(monsters_dict[monster][-500:]))
    monsters_dict[monster] = re.sub(
        r'<div class="section15">.*$',
        '', 
        monsters_dict[monster], 
        flags=re.DOTALL
    )
        
monster_text = monsters_dict["outsiders_demon_balor"]
print(monster_text)



<p class="title">
     Balor
     <span class="level">
      CR 20
     </span>
    </p>
    <p>
     <b>
      XP 307,200
     </b>
     <br/>
     CE Large
     <a href="https://www.d20pfsrd.com/bestiary/rules-for-monsters/creature-types#Outsider" rel="nofollow">
      outsider
     </a>
     (
     <a href="https://www.d20pfsrd.com/bestiary/rules-for-monsters/creature-types#TOC-Chaotic" rel="nofollow">
      chaotic
     </a>
     ,
     <a href="https://www.d20pfsrd.com/bestiary/rules-for-monsters/creature-types#TOC-Demon" rel="nofollow">
      demon
     </a>
     ,
     <a href="https://www.d20pfsrd.com/bestiary/rules-for-monsters/creature-types#TOC-Evil" rel="nofollow">
      evil
     </a>
     ,
     <a href="https://www.d20pfsrd.com/bestiary/rules-for-monsters/creature-types#extraplanar" rel="nofollow">
      extraplanar
     </a>
     )
     <br/>
     <b>
      Init
     </b>
     +11;
     <b>
      Senses
     </b>
     <a href="https://www.d20pfsrd.com/gamemastering/spec

#### next

In [2193]:
monster_text = monsters_dict["outsiders_demon_balor"]
print(monster_text)

<p class="title">
     Balor
     <span class="level">
      CR 20
     </span>
    </p>
    <p>
     <b>
      XP 307,200
     </b>
     <br/>
     CE Large
     <a href="https://www.d20pfsrd.com/bestiary/rules-for-monsters/creature-types#Outsider" rel="nofollow">
      outsider
     </a>
     (
     <a href="https://www.d20pfsrd.com/bestiary/rules-for-monsters/creature-types#TOC-Chaotic" rel="nofollow">
      chaotic
     </a>
     ,
     <a href="https://www.d20pfsrd.com/bestiary/rules-for-monsters/creature-types#TOC-Demon" rel="nofollow">
      demon
     </a>
     ,
     <a href="https://www.d20pfsrd.com/bestiary/rules-for-monsters/creature-types#TOC-Evil" rel="nofollow">
      evil
     </a>
     ,
     <a href="https://www.d20pfsrd.com/bestiary/rules-for-monsters/creature-types#extraplanar" rel="nofollow">
      extraplanar
     </a>
     )
     <br/>
     <b>
      Init
     </b>
     +11;
     <b>
      Senses
     </b>
     <a href="https://www.d20pfsrd.com/gamemastering/spec

In [2196]:
# for monster in monsters_dict:
#     top_pattern = r'<p.*?>(.*?)</p>'
#     part_1 = re.search(top_pattern, monsters_dict[monster], flags=re.DOTALL).group(0)
#     part_1 = re.sub(r'<[^>]+>', '', part_1).strip()
#     part_1 = re.sub(r'\s+', ' ', part_1).strip()
#     monsters_data[monster]['top_'] = part_1
#     monsters_dict[monster] = re.sub(top_pattern, '', monsters_dict[monster], flags=re.DOTALL)
#     print(part_1)
#     # monsters_data[monster]['source'] = source

for monster in monsters_dict:
    top_pattern = r'<p class="(?:title|header|creature)".*?(<span(?: class="(?:level|creature-level)")?)?.*?</p>'
    match = re.search(top_pattern, monsters_dict[monster], flags=re.DOTALL)
    if match:
        top_ = match.group(0)
        monsters_dict[monster] = monsters_dict[monster].replace(top_, '', 1)
        top_ = re.sub(r'<[^>]+>', '', top_).strip()
        top_ = re.sub(r'\s+', ' ', top_).strip()
        monsters_data[monster]['top_'] = top_
        print(top_)

The Pale Horse of Death
Lemure [Augmented*] CR 1
Baboon Companions
Tylosaurus Companions
DEFENSE
Wereshark (Hybrid Form) CR 3
Wielding Spear Two-Handed:
DEFENSE
Weremantis (Hybrid Form) CR 4
DEFENSE
DEFENSE
Pod-Spawned Guard Captain CR 8
Parasaurolophus Companions
DEFENSE
Weretiger (Hybrid Form) CR 4
Cobra-Back Inphidian Characters
Manta Ray
Weaponized Carrion Golem
DEFENSE
Werecrocodile (Hybrid Form)
Werewolf (Hybrid Form) CR 2
DEFENSE
Carrion Golem
DEFENSE
Mortuary Cyclone Spawn
Wereboar (Hybrid Form) CR 2
Wererat (Hybrid Form) CR 2
DEFENSE
OFFENSE
MOUSE LORD (HUMAN FORM) CR 13
DEFENSE
Nilbogs as Characters
Cat Lord (Human Form) CR 15
DEFENSE
ACTIONS
DEFENSE
Frog, Poisonous (Giant) CR 1
Carrion Golem
DEFENSE
DEFENSE
ABOUT
Starting Statistics
Elephant/Mastodon Companions
DEFENSE
DEFENSE
DEFENSE
DEFENSE
DEFENSE
DEFENSE
Rattler Inphidian Characters
Clockwork Golem
Spriggan (Large size) CR 3
DEFENSE
DEFENSE
DEFENSE
Carrion Golem (Stand-In)
Giant Moray Eel Companions
DEFENSE
DEFENSE
DEFEN

In [2197]:
monster_text = monsters_dict["outsiders_demon_balor"]
print(monsters_data["outsiders_demon_balor"])
print(monster_text)

{'title1': 'Demon, Balor', 'title2': 'balor', 'desc_short': 'This winged fiend’s horned head and fanged visage present the perfection of the demonic form, fire spurting from its flesh.', 'source': '', 'top_': 'Balor CR 20'}

    <p>
     <b>
      XP 307,200
     </b>
     <br/>
     CE Large
     <a href="https://www.d20pfsrd.com/bestiary/rules-for-monsters/creature-types#Outsider" rel="nofollow">
      outsider
     </a>
     (
     <a href="https://www.d20pfsrd.com/bestiary/rules-for-monsters/creature-types#TOC-Chaotic" rel="nofollow">
      chaotic
     </a>
     ,
     <a href="https://www.d20pfsrd.com/bestiary/rules-for-monsters/creature-types#TOC-Demon" rel="nofollow">
      demon
     </a>
     ,
     <a href="https://www.d20pfsrd.com/bestiary/rules-for-monsters/creature-types#TOC-Evil" rel="nofollow">
      evil
     </a>
     ,
     <a href="https://www.d20pfsrd.com/bestiary/rules-for-monsters/creature-types#extraplanar" rel="nofollow">
      extraplanar
     </a>
     )
    